# LCO / Si-C 复合负极电池 — 倍率性能仿真

**使用参数文件**: `userdata/lithum_ion/Custom_LCO_SiC.py`  
**模型**: DFN (Doyle-Fuller-Newman) + 双相复合负极  
**正极**: LCO (钴酸锂)  
**负极**: 石墨 (Primary) + 硅碳 (Secondary)  

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import pybamm

plt.rcParams["font.family"] = "Microsoft YaHei"
plt.rcParams["axes.unicode_minus"] = False

print(f"PyBaMM version: {pybamm.__version__}")

## 1. 加载自定义参数

In [ ]:
# 从自定义用户文件导入参数
from lithum_ion.Custom_LCO_SiC import get_parameter_values

params = pybamm.ParameterValues(get_parameter_values())

# 查看参数数量
print(f"参数总数: {len(params.store)}")
print("\n已加载的相专属参数示例:")
for key in sorted(params.store.keys()):
    if "Primary: Maximum" in key or "Secondary: Maximum" in key:
        print(f"  {key} = {params[key]}")
    if (
        "Primary: Negative electrode active" in key
        or "Secondary: Negative electrode active" in key
    ):
        print(f"  {key} = {params[key]}")

## 2. 创建复合电极模型

使用 `"particle phases": ("2", "1")` 选项:
- 负极: 2 相 (石墨 Primary + 硅碳 Secondary)
- 正极: 1 相 (LCO)

In [ ]:
# 选择模型选项
options = {
    "particle phases": ("2", "1"),  # 负极2相, 正极1相
    "thermal": "lumped",  # 集总热模型
}

model = pybamm.lithium_ion.DFN(options)
print(f"模型类型: {model.name}")
print(f"模型选项: {model.options}")

## 3. 倍率性能仿真

在不同 C-rate 下进行恒流放电, 对比电压曲线和容量。

In [ ]:
# 定义要测试的倍率 (精简为 4 个, 加速演示)
C_rates = [0.2, 0.5, 1.0, 2.0]

# 标称容量 [Ah]
nominal_capacity = params["Nominal cell capacity [A.h]"]
print(f"标称容量: {nominal_capacity} Ah")

# 复合负极缺少整极密度参数 → 用两相体积分数加权平均补上
if "Negative electrode density [kg.m-3]" not in params:
    rho_p = params["Primary: Negative electrode density [kg.m-3]"]
    rho_s = params["Secondary: Negative electrode density [kg.m-3]"]
    vf_p = params["Primary: Negative electrode active material volume fraction"]
    vf_s = params["Secondary: Negative electrode active material volume fraction"]
    rho_avg = (vf_p * rho_p + vf_s * rho_s) / (vf_p + vf_s)
    params["Negative electrode density [kg.m-3]"] = rho_avg
    print(f"补充参数 Negative electrode density = {rho_avg:.1f} kg/m³ (加权平均)")

# 存储仿真结果
solutions = {}
import time

total = len(C_rates)
t_start_all = time.time()

for idx, C in enumerate(C_rates):
    current = C * nominal_capacity  # [A]

    # 更新电流参数
    params.update({"Current function [A]": current})

    # 创建实验条件: 恒流放电至截止电压
    experiment = pybamm.Experiment(
        [f"Discharge at {current}A until 3.0 V"],
    )

    # 运行仿真 — safe 模式 + 优化 solver 参数应对双相 DFN 刚性
    solver = pybamm.CasadiSolver(
        mode="safe", dt_max=30, max_step_decrease_count=8, rtol=1e-4, atol=1e-4
    )
    # 降低网格密度加速 (演示用)
    var_pts = {
        "x_n": 10,
        "x_s": 10,
        "x_p": 10,
        "r_n": 10,
        "r_p": 10,
        "r_n_prim": 10,
        "r_n_sec": 10,
        "r_p_prim": 10,
    }
    sim = pybamm.Simulation(
        model,
        experiment=experiment,
        parameter_values=params,
        solver=solver,
        var_pts=var_pts,
    )

    t0 = time.time()
    print(
        f"[{idx + 1}/{total}] 求解 {C}C (电流 {current:.2f} A)...", end=" ", flush=True
    )
    solution = sim.solve()
    elapsed = time.time() - t0
    solutions[C] = solution

    # 提取放电容量
    cap = solution["Discharge capacity [A.h]"].data[-1]

    # 预估剩余时间
    avg_time = (time.time() - t_start_all) / (idx + 1)
    eta = avg_time * (total - idx - 1)
    print(f"容量 {cap:.4f} Ah | 耗时 {elapsed:.1f}s | 剩余 ≈ {eta:.0f}s")

total_time = time.time() - t_start_all
print(f"\n全部倍率仿真完成! 总耗时 {total_time:.1f}s")

## 4. 结果可视化

### 4.1 不同倍率下的放电电压曲线

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.get_cmap("viridis")(np.linspace(0.1, 0.9, len(C_rates)))

for i, C in enumerate(C_rates):
    sol = solutions[C]
    cap = sol["Discharge capacity [A.h]"].data
    voltage = sol["Terminal voltage [V]"].data
    ax.plot(cap, voltage, color=colors[i], lw=1.8, label=f"{C}C")

ax.set_xlabel("Discharge Capacity [Ah]", fontsize=12)
ax.set_ylabel("Terminal Voltage [V]", fontsize=12)
ax.set_title("LCO / Si-C 复合负极 — 倍率放电曲线", fontsize=14, fontweight="bold")
ax.legend(title="C-rate", loc="lower left", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, None)

# 标注截止电压
ax.axhline(y=3.0, color="red", ls="--", alpha=0.5, label="Cut-off (3.0V)")

plt.tight_layout()
plt.show()

### 4.2 容量保持率 vs 倍率 (Rate Capability)

In [ ]:
capacities = []
for C in C_rates:
    cap = solutions[C]["Discharge capacity [A.h]"].data[-1]
    capacities.append(cap)

# 以 0.2C 为基准计算保持率
base_cap = capacities[0]
retention = [c / base_cap * 100 for c in capacities]

fig, ax1 = plt.subplots(figsize=(8, 5))

bars = ax1.bar(
    range(len(C_rates)), capacities, color=colors, edgecolor="black", linewidth=0.5
)
ax1.set_xticks(range(len(C_rates)))
ax1.set_xticklabels([f"{C}C" for C in C_rates])
ax1.set_ylabel("Discharge Capacity [Ah]", fontsize=12)
ax1.set_xlabel("C-rate", fontsize=12)
ax1.set_title("LCO / Si-C — 容量保持率 vs 倍率", fontsize=14, fontweight="bold")

# 在柱状图上标注保持率
for bar, ret in zip(bars, retention):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{ret:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

ax1.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 50)
print("容量保持率汇总:")
print("=" * 50)
for C, cap, ret in zip(C_rates, capacities, retention):
    print(f"  {C:>4.1f}C  →  {cap:.4f} Ah  ({ret:.1f}%)")

### 4.3 各相反应电流贡献 (了解 Primary/Secondary 相分担)

In [ ]:
C_plot = 1.0  # 选择 1C 来查看
sol = solutions[C_plot]

fig, ax = plt.subplots(figsize=(10, 4))

try:  # 使用 X-averaged 变量: 空间平均后仅随时间变化 (1D), 与电压维度匹配
    j_prim = sol[
        "X-averaged negative electrode primary volumetric interfacial current density [A.m-3]"
    ]
    j_sec = sol[
        "X-averaged negative electrode secondary volumetric interfacial current density [A.m-3]"
    ]

    # X-averaged 变量是时间序列，直接取 data
    j_prim_data = j_prim.data / 1e6  # 转为 mA/cm³
    j_sec_data = j_sec.data / 1e6
    voltage = sol["Terminal voltage [V]"].data

    ax.plot(voltage, j_prim_data, label="Primary (Graphite)", lw=2)
    ax.plot(voltage, j_sec_data, label="Secondary (Silicon)", lw=2)
    ax.set_xlabel("Terminal Voltage [V]", fontsize=12)
    ax.set_ylabel("Volumetric Current Density [mA/cm³]", fontsize=12)
    ax.set_title(
        f"Primary vs Secondary 反应电流 @ {C_plot}C", fontsize=12, fontweight="bold"
    )
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
except Exception as e:
    print(f"注意: 反应电流分量提取失败 ({e})")
    print("这可能是因为模型输出变量名在不同版本中有差异。")
    print("\n可用变量列表 (部分):")
    # 列出包含 'primary' 或 'secondary' 的变量名
    for var in sorted(sol.all_models[0].variables.keys()):
        if "primary" in var.lower() or "secondary" in var.lower():
            print(f"  {var}")

plt.tight_layout()
plt.show()

### 4.4 温度变化 (不同倍率下的温升)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for i, C in enumerate(C_rates):
    sol = solutions[C]
    try:
        T_raw = sol["Cell temperature [K]"].data  # 可能为 (spatial, time) 或 (time,)
        T = T_raw - 273.15  # 转为 °C
        cap = sol["Discharge capacity [A.h]"].data
        # 如果 T 是多维的, 取空间平均使其变为 1D
        if T.ndim > 1:
            T = T.mean(axis=0)
        ax.plot(cap, T, color=colors[i], lw=1.8, label=f"{C}C")
    except Exception:
        ax.plot([], [])

ax.set_xlabel("Discharge Capacity [Ah]", fontsize=12)
ax.set_ylabel("Cell Temperature [°C]", fontsize=12)
ax.set_title("LCO / Si-C — 倍率放电温升", fontsize=14, fontweight="bold")
ax.legend(title="C-rate", fontsize=10)
ax.grid(True, alpha=0.3)

# 标注环境温度
T_amb = params["Ambient temperature [K]"] - 273.15
ax.axhline(y=T_amb, color="gray", ls="--", alpha=0.7, label=f"Ambient ({T_amb:.0f}°C)")
ax.legend(title="C-rate", fontsize=10)

plt.tight_layout()
plt.show()

## 5. 结果总结

- 倍率性能取决于多个因素: 颗粒粒径、扩散系数、电导率等
- 硅相在大倍率下贡献可能下降 (动力学和扩散限制)
- **如果仿真结果与实验不符**: 请检查参数文件中的 OCP、扩散系数、交换电流密度是否为占位值
- **关键待测参数** (见 `参数收集说明.md`):
  1. 正极 LCO OCP → 半电池实验
  2. 负极石墨 OCP → 半电池实验
  3. 负极硅相 lithiation/delithiation OCP → 硅半电池实验
  4. 各相扩散系数 → GITT / EIS
  5. 电解液电导率/扩散系数 → 文献或实测

## 6. 硅负极体积膨胀仿真

PyBaMM 支持通过 `"particle mechanics"` 选项模拟嵌锂引起的颗粒体积膨胀。

**模型选项**:
- `"swelling only"` — 仅模拟体积膨胀（应力+位移+厚度变化）
- `"swelling and cracking"` — 同时模拟裂纹扩展

**本节目标**: 在不同倍率放电下，观察硅相 (Secondary) 的体积膨胀程度、电极厚度变化以及表面应力。

**关键参数** (需要补充):
| 参数 | 含义 | 硅典型值 |
|------|------|----------|
| `Secondary: Negative electrode volume change` | 硅相体积变化函数 $\Delta V / V$ | 需自定义 |
| `Secondary: Negative electrode partial molar volume [m3.mol-1]` | Li 在硅中的偏摩尔体积 | ~8.5e-6 |
| `Secondary: Negative electrode Young's modulus [Pa]` | 杨氏模量 | ~40 GPa |
| `Negative electrode Poisson's ratio` | 泊松比 | ~0.28 |

In [ ]:
# --- 6.1 定义硅相体积变化函数 ---
# 硅嵌锂时体积膨胀 ~300% (Li15Si4), 体积变化与化学计量比近似线性关系
# 参考: Obrovac & Chevrier, Chem. Rev. 2014; Becker et al., JES 2019


def graphite_volume_change_Ai2020(sto):
    """石墨体积变化 (Ai2020 多项式拟合)"""
    p1, p2, p3, p4 = 145.907, -681.229, 1334.442, -1415.710
    p5, p6, p7, p8 = 873.906, -312.528, 60.641, -5.706
    p9, p10 = 0.386, -4.966e-05
    t_change = (
        p1 * sto**9
        + p2 * sto**8
        + p3 * sto**7
        + p4 * sto**6
        + p5 * sto**5
        + p6 * sto**4
        + p7 * sto**3
        + p8 * sto**2
        + p9 * sto
        + p10
    )
    return t_change


def silicon_volume_change(sto):
    """
    硅相体积变化函数 (简化线性模型).

    硅嵌锂时体积膨胀最大约 300% (对应 Li₃.₇₅Si, 即 sto≈1)
    使用二次插值使初始斜率为零 (更合理).

    Parameters
    ----------
    sto : float or array
        硅相化学计量比 (0 = 完全脱锂, 1 = 完全嵌锂)

    Returns
    -------
    float or array
        体积变化比 ΔV/V (相对于脱锂态)
    """
    # 线性模型: 完全嵌锂时膨胀 ~300%
    # 嵌锂化学计量比 y = sto, 体积膨胀 ~ 1.2 * sto
    return 1.2 * sto


print("硅相体积变化函数已定义 (线性近似, 最大膨胀 ~120%)")
print("注: 可根据实验数据替换为更精确的多项式或插值函数")

### 6.2 创建含膨胀的模型并运行仿真

In [ ]:
# --- 6.2 创建含颗粒力学效应的模型并运行仿真 ---

# 创建含 swelling 的模型
options_swelling = {
    "particle phases": ("2", "1"),  # 负极 2 相, 正极 1 相
    "thermal": "lumped",
    "particle mechanics": ("swelling only", "none"),  # 负极膨胀, 正极无力学
    "stress-induced diffusion": "false",  # 关闭应力诱导扩散, 减少参数需求
}

model_sw = pybamm.lithium_ion.DFN(options_swelling)
print(f"膨胀模型: {model_sw.name}")

# 复制参数并添加力学相关参数
from lithum_ion.Custom_LCO_SiC import get_parameter_values

params_sw = pybamm.ParameterValues(get_parameter_values())

# 补充密度参数
if "Negative electrode density [kg.m-3]" not in params_sw:
    rho_p = params_sw["Primary: Negative electrode density [kg.m-3]"]
    rho_s = params_sw["Secondary: Negative electrode density [kg.m-3]"]
    vf_p = params_sw["Primary: Negative electrode active material volume fraction"]
    vf_s = params_sw["Secondary: Negative electrode active material volume fraction"]
    params_sw["Negative electrode density [kg.m-3]"] = (vf_p * rho_p + vf_s * rho_s) / (
        vf_p + vf_s
    )

# ------ 添加 Primary 相 (石墨) 力学参数 (参考 Ai2020) ------
params_sw.update(
    {
        "Primary: Negative electrode volume change": graphite_volume_change_Ai2020,
        "Primary: Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
        "Primary: Negative electrode Young's modulus [Pa]": 15e9,
        "Primary: Negative electrode Poisson's ratio": 0.3,
        "Primary: Negative electrode reference concentration for free of deformation [mol.m-3]": 0.0,
    }
)

# ------ 添加 Secondary 相 (硅) 力学参数 ------
params_sw.update(
    {
        "Secondary: Negative electrode volume change": silicon_volume_change,
        "Secondary: Negative electrode partial molar volume [m3.mol-1]": 8.5e-6,
        "Secondary: Negative electrode Young's modulus [Pa]": 40e9,
        "Secondary: Negative electrode Poisson's ratio": 0.28,
        "Secondary: Negative electrode reference concentration for free of deformation [mol.m-3]": 0.0,
    }
)

print("力学参数已添加完毕")

# 运行 1C 放电
nominal_capacity = params_sw["Nominal cell capacity [A.h]"]
C_sw = 1.0
current_sw = C_sw * nominal_capacity
params_sw.update({"Current function [A]": current_sw})

experiment_sw = pybamm.Experiment(
    [f"Discharge at {current_sw}A until 3.0 V"],
)

solver_sw = pybamm.CasadiSolver(
    mode="safe", dt_max=30, max_step_decrease_count=8, rtol=1e-4, atol=1e-4
)

var_pts_sw = {
    "x_n": 10,
    "x_s": 10,
    "x_p": 10,
    "r_n": 10,
    "r_p": 10,
    "r_n_prim": 10,
    "r_n_sec": 10,
    "r_p_prim": 10,
}
sim_sw = pybamm.Simulation(
    model_sw,
    experiment=experiment_sw,
    parameter_values=params_sw,
    solver=solver_sw,
    var_pts=var_pts_sw,
)

print(f"\n求解含膨胀效应的 {C_sw}C 放电...")
sol_sw = sim_sw.solve()
print("求解完成!")
print(f"放电容量: {sol_sw['Discharge capacity [A.h]'].data[-1]:.4f} Ah")

### 6.3 膨胀结果可视化

In [ ]:
# --- 6.3 绘制硅负极膨胀相关结果 ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) 负极各相颗粒表面位移 (膨胀量)
ax = axes[0, 0]
try:
    disp_prim = sol_sw[
        "X-averaged negative electrode primary particle surface displacement [m]"
    ].data
    disp_sec = sol_sw[
        "X-averaged negative electrode secondary particle surface displacement [m]"
    ].data
    t = sol_sw["Time [s]"].data / 3600  # 转为小时

    ax.plot(t, disp_prim * 1e6, label="Primary (Graphite)", lw=2)
    ax.plot(t, disp_sec * 1e6, label="Secondary (Silicon)", lw=2, color="red")
    ax.set_xlabel("Time [h]")
    ax.set_ylabel("Surface Displacement [μm]")
    ax.set_title("(a) 颗粒表面位移 (膨胀)", fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
except Exception as e:
    ax.text(
        0.5, 0.5, f"提取失败:\n{e}", ha="center", va="center", transform=ax.transAxes
    )

# (b) 负极电极厚度变化
ax = axes[0, 1]
try:
    # 总电极厚度变化
    dthick = sol_sw["Negative electrode thickness change [m]"].data
    # 各相贡献
    dthick_p = sol_sw["X-averaged negative electrode primary thickness change [m]"].data
    dthick_s = sol_sw[
        "X-averaged negative electrode secondary thickness change [m]"
    ].data
    t = sol_sw["Time [s]"].data / 3600

    ax.plot(t, dthick_p * 1e6, label="Primary (Graphite)", lw=2)
    ax.plot(t, dthick_s * 1e6, label="Secondary (Silicon)", lw=2, color="red")
    ax.plot(t, dthick * 1e6, label="Total", lw=2, ls="--", color="black")
    ax.set_xlabel("Time [h]")
    ax.set_ylabel("Thickness Change [μm]")
    ax.set_title("(b) 负极各相厚度变化", fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
except Exception as e:
    ax.text(
        0.5, 0.5, f"提取失败:\n{e}", ha="center", va="center", transform=ax.transAxes
    )

# (c) 硅相化学计量比 vs 体积变化
ax = axes[1, 0]
try:
    sto_sec = sol_sw["X-averaged negative secondary particle stoichiometry"].data
    if sto_sec.ndim > 1:
        sto_sec = sto_sec.mean(axis=0)
    # 硅相体积变化 (通过自定义函数估算)
    vol_change = silicon_volume_change(sto_sec) * 100  # 转为 %
    t = sol_sw["Time [s]"].data / 3600

    ax.plot(t, vol_change, color="red", lw=2)
    ax.set_xlabel("Time [h]")
    ax.set_ylabel("Volume Change [%]")
    ax.set_title("(c) 硅相体积膨胀", fontweight="bold")
    ax.grid(True, alpha=0.3)

    # 标注最大膨胀
    max_idx = np.argmax(vol_change)
    ax.annotate(
        f"Max: {vol_change[max_idx]:.1f}%",
        xy=(t[max_idx], vol_change[max_idx]),
        xytext=(t[max_idx] * 0.6, vol_change[max_idx] * 0.8),
        arrowprops=dict(arrowstyle="->", color="red"),
        fontsize=11,
        color="red",
        fontweight="bold",
    )
except Exception as e:
    ax.text(
        0.5, 0.5, f"提取失败:\n{e}", ha="center", va="center", transform=ax.transAxes
    )

# (d) 负极总厚度变化
ax = axes[1, 1]
try:
    dthick_neg = sol_sw["Negative electrode thickness change [m]"].data
    if dthick_neg.ndim > 1:
        dthick_neg = dthick_neg.mean(axis=0)
    t = sol_sw["Time [s]"].data / 3600

    ax.plot(t, dthick_neg * 1e6, color="black", lw=2)
    ax.fill_between(t, 0, dthick_neg * 1e6, alpha=0.2, color="blue")
    ax.set_xlabel("Time [h]")
    ax.set_ylabel("Negative Electrode Thickness Change [μm]")
    ax.set_title("(d) 负极总厚度变化", fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color="gray", ls="-", alpha=0.5)
except Exception as e:
    ax.text(
        0.5, 0.5, f"提取失败:\n{e}", ha="center", va="center", transform=ax.transAxes
    )

plt.suptitle(
    "LCO / Si-C — 硅负极体积膨胀分析 @ 1C 放电", fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

### 6.4 不同倍率下的硅膨胀对比

In [ ]:
# --- 6.4 不同倍率下的硅相膨胀对比 ---
C_rates_sw = [0.2, 1.0, 3.0]
colors_sw = plt.get_cmap("coolwarm")(np.linspace(0.1, 0.9, len(C_rates_sw)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, C in enumerate(C_rates_sw):
    curr = C * nominal_capacity
    params_sw.update({"Current function [A]": curr})

    exp = pybamm.Experiment([f"Discharge at {curr}A until 3.0 V"])
    sim_i = pybamm.Simulation(
        model_sw,
        experiment=exp,
        parameter_values=params_sw,
        solver=pybamm.CasadiSolver(
            mode="safe", dt_max=30, max_step_decrease_count=8, rtol=1e-4, atol=1e-4
        ),
        var_pts=var_pts_sw,
    )
    sol_i = sim_i.solve()

    try:
        # 硅相化学计量比 (可能含空间维度, 需取平均)
        sto_raw = sol_i["X-averaged negative secondary particle stoichiometry"].data
        if sto_raw.ndim > 1:
            sto_raw = sto_raw.mean(axis=0)
        vol_pct = silicon_volume_change(sto_raw) * 100
        t = sol_i["Time [s]"].data / 3600

        axes[0].plot(t, vol_pct, color=colors_sw[idx], lw=2, label=f"{C}C")
        axes[0].set_xlabel("Time [h]", fontsize=12)
        axes[0].set_ylabel("Silicon Volume Change [%]", fontsize=12)
        axes[0].set_title("硅相体积膨胀 vs 时间", fontsize=13, fontweight="bold")
        axes[0].legend(title="C-rate", fontsize=10)
        axes[0].grid(True, alpha=0.3)
    except Exception as e:
        print(f"{C}C 膨胀数据提取失败: {e}")

    try:
        dthick_total = sol_i["Negative electrode thickness change [m]"].data
        if dthick_total.ndim > 1:
            dthick_total = dthick_total.mean(axis=0)
        t = sol_i["Time [s]"].data / 3600

        axes[1].plot(t, dthick_total * 1e6, color=colors_sw[idx], lw=2, label=f"{C}C")
        axes[1].set_xlabel("Time [h]", fontsize=12)
        axes[1].set_ylabel("Negative Electrode Thickness Change [μm]", fontsize=12)
        axes[1].set_title("负极厚度变化 vs 时间", fontsize=13, fontweight="bold")
        axes[1].legend(title="C-rate", fontsize=10)
        axes[1].grid(True, alpha=0.3)
    except Exception as e:
        print(f"{C}C 厚度数据提取失败: {e}")

plt.suptitle(
    "LCO / Si-C — 不同倍率下硅膨胀对比", fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

## 7. 膨胀仿真总结

**已添加的力学参数**:
- `Primary: Negative electrode volume change` — 石墨膨胀函数 (Ai2020 多项式)
- `Secondary: Negative electrode volume change` — 硅膨胀函数 (线性近似)
- `Primary: Negative electrode partial molar volume [m3.mol-1]` — 3.1e-6
- `Secondary: Negative electrode partial molar volume [m3.mol-1]` — 8.5e-6
- `Primary/Secondary: Negative electrode Young's modulus [Pa]` — 15/40 GPa
- `Negative electrode Poisson's ratio` — 0.28

**注意事项**:
1. 上述硅膨胀函数为**线性简化模型**。实际硅的体积膨胀与嵌锂量呈复杂非线性关系，建议用实验数据替换
2. 可以将 `"particle mechanics"` 改为 `"swelling and cracking"` 来进一步模拟裂纹扩展，但需要额外参数: 初始裂纹长度、裂纹速率等
3. 参数 `"stress-induced diffusion": "true"` 可启用应力诱导扩散效应，使膨胀-扩散耦合更加真实